# NCBI Entrez Tools

![NCBI Entrez Tools](https://proto-bio.github.io/proto-assets/images/tool/ncbi/hero.png)

This notebook demonstrates the three NCBI E-utility wrappers provided by proto_tools. The `run_ncbi_esearch` function finds database record IDs matching a query term, `run_ncbi_esummary` retrieves structured metadata for a given record, and `run_ncbi_efetch` downloads full sequences in FASTA format. Together, these three functions support a common bioinformatics workflow: search for records of interest, inspect their metadata, and then fetch the actual sequence data for downstream analysis. A fourth function, `run_ncbi_reference_bundle`, downloads a whole version-pinned genome assembly from NCBI Datasets.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("ncbi")
display_overview("ncbi")
display_docs_section("ncbi", "Background")

## Available tools

In [ ]:
display_available_tools("ncbi")

### `run_ncbi_esearch`

The first step in most NCBI workflows is finding the database IDs for records of interest. `run_ncbi_esearch` accepts a free-text query using NCBI's Entrez search syntax, which supports field-qualified terms like gene names and organism filters. It returns a list of matching record IDs from the protein, nuccore, or gene database, which can then be passed directly to `run_ncbi_esummary` or `run_ncbi_efetch` for further retrieval.

In [ ]:
from proto_tools import (
    NCBIEsearchInput,
    NCBIEsearchConfig,
    run_ncbi_esearch,
)

In [ ]:
# Display docs
display_api_reference("ncbi", "input", "run_ncbi_esearch")

# Search for lac repressor protein IDs in Escherichia coli
inputs = NCBIEsearchInput(
    db="protein",
    search_term='"lacI"[Gene] AND "Escherichia coli"[Organism]',
    max_results=5,
)

In [ ]:
# Display config docs
display_api_reference("ncbi", "config", "run_ncbi_esearch")

# Each NCBI function has its own config class | see docs above for all fields
config = NCBIEsearchConfig()

In [ ]:
# Run the search
search_result = run_ncbi_esearch(inputs, config)

In [ ]:
# Display output docs
display_api_reference("ncbi", "output", "run_ncbi_esearch")

# Inspect the returned IDs
print(f"IDs found: {search_result.ids}")

### `run_ncbi_esummary`

Once you have record IDs from a search, `run_ncbi_esummary` retrieves structured metadata without downloading the full record. This is useful for inspecting gene annotations, genomic coordinates, organism information, and other catalog-level details. It accepts a database name and a single accession or NCBI ID, and returns the full summary document as a parsed dictionary.

In [ ]:
from proto_tools import (
    NCBIEsummaryInput,
    NCBIEsummaryConfig,
    run_ncbi_esummary,
)

In [ ]:
# Display docs
display_api_reference("ncbi", "input", "run_ncbi_esummary")

# Fetch the gene summary for human TP53 using its NCBI Gene ID
esummary_inputs = NCBIEsummaryInput(
    db="gene",
    identifier="7157",
)

In [ ]:
# Display config docs
display_api_reference("ncbi", "config", "run_ncbi_esummary")

# Each NCBI function has its own config class | see docs above for all fields
esummary_config = NCBIEsummaryConfig()

In [ ]:
# Run the summary fetch
summary_result = run_ncbi_esummary(esummary_inputs, esummary_config)

In [ ]:
# Display output docs
display_api_reference("ncbi", "output", "run_ncbi_esummary")

# Inspect gene metadata for TP53
gene_id = esummary_inputs.identifier
gene_data = summary_result.summary.get(gene_id, {})
print(f"Name: {gene_data.get('name')}")
print(f"Description: {gene_data.get('description')}")
print(f"Organism: {gene_data.get('organism', {}).get('scientificname')}")

genomic_info = gene_data.get("genomicinfo", [])
if genomic_info:
    region = genomic_info[0]
    print(f"Chromosome: {region.get('chraccver')}")
    print(f"Coordinates: {region.get('chrstart')} to {region.get('chrstop')}")

### `run_ncbi_efetch`

`run_ncbi_efetch` downloads full sequence records from NCBI in FASTA format. It is the workhorse for retrieving protein or nucleotide sequences needed for alignment, structure prediction, or other downstream analyses. In addition to full-record retrieval, it supports subsequence extraction by specifying start and stop coordinates (1-indexed, inclusive), and strand selection for nucleotide records, making it suitable for pulling specific gene loci or regulatory elements from large chromosome assemblies.

In [ ]:
from proto_tools import (
    NCBIEfetchInput,
    NCBIEfetchConfig,
    run_ncbi_efetch,
)

In [ ]:
# Display docs
display_api_reference("ncbi", "input", "run_ncbi_efetch")

# Fetch the TP53 protein sequence by RefSeq accession
efetch_inputs = NCBIEfetchInput(
    db="protein",
    identifier="NP_000537.3",
    return_format="fasta",
)

In [ ]:
# Display config docs
display_api_reference("ncbi", "config", "run_ncbi_efetch")

# Each NCBI function has its own config class | see docs above for all fields
efetch_config = NCBIEfetchConfig()

In [ ]:
# Run the fetch
fetch_result = run_ncbi_efetch(efetch_inputs, efetch_config)

In [ ]:
# Display output docs
display_api_reference("ncbi", "output", "run_ncbi_efetch")

# Inspect the fetched FASTA record
record = fetch_result.fasta_records[0]
print(f"Accession: {record.accession}")
print(f"Header: {record.header[:80]}...")
print(f"Sequence length: {len(record.sequence)}")
print(f"Preview: {record.sequence[:60]}...")

#### Genomic Subsequence Extraction

When working with large genomes, you often need just a small region rather than the entire chromosome. The efetch function supports coordinate-based extraction with strand selection, which is useful for pulling promoter regions, specific gene loci, or regulatory elements. Here we retrieve a 201 bp region from the lacI locus on the minus strand of the *E. coli* K-12 genome.

In [ ]:
# Fetch a 201 bp region from the lacI locus (minus strand)
subseq_inputs = NCBIEfetchInput(
    db="nuccore",
    identifier="NC_000913.3",
    return_format="fasta",
    seq_start=366428,
    seq_stop=366628,
    strand="-",
)

subseq_result = run_ncbi_efetch(subseq_inputs, efetch_config)

record = subseq_result.fasta_records[0]
print(f"Accession: {record.accession}")
print(f"Sequence length: {len(record.sequence)}")
print(f"Preview: {record.sequence[:80]}...")
print(f"Source URL: {subseq_result.source_url}")

### `run_ncbi_reference_bundle`

`run_ncbi_reference_bundle` downloads the NCBI Datasets bundle for a versioned assembly accession and returns a summary of the assembly with paths to its files: genome FASTA, GFF3 annotation, transcript FASTA, and protein FASTA by default, with GenBank, GTF, and CDS FASTA available through `annotation_types`. By default the bundle is stored under `$PROTO_DATABASES_DIR/ncbi/<accession>/` and reused by later calls; here we extract the E. coli K-12 MG1655 reference (about 3 MB) into a temporary directory instead.

In [ ]:
from tempfile import TemporaryDirectory

from proto_tools import (
    NCBIReferenceBundleConfig,
    NCBIReferenceBundleInput,
    run_ncbi_reference_bundle,
)

bundle_dir = TemporaryDirectory()

In [ ]:
# Display docs
display_api_reference("ncbi", "input", "run_ncbi_reference_bundle")

# A versioned accession; an NCBI Datasets page link works too
bundle_inputs = NCBIReferenceBundleInput(accession="GCF_000005845.2", destination_dir=bundle_dir.name)

In [ ]:
# Display config docs
display_api_reference("ncbi", "config", "run_ncbi_reference_bundle")

# Default config downloads all four annotation types | see docs above for all fields
bundle_config = NCBIReferenceBundleConfig()

In [ ]:
# Download and extract the bundle
bundle = run_ncbi_reference_bundle(bundle_inputs, bundle_config)

In [ ]:
from pathlib import Path

# Display output docs
display_api_reference("ncbi", "output", "run_ncbi_reference_bundle")

# What NCBI says this assembly is
assembly = bundle.assembly
print(f"{assembly.organism_name} (taxid {assembly.tax_id})")
print(f"{assembly.assembly_name}, {assembly.assembly_level}: {assembly.total_sequence_length:,} bp, {assembly.gc_percent}% GC\n")

# Each path is None when the assembly has no such file or it was not requested
for field in ("genome_fasta", "genome_gff", "rna_fasta", "protein_fasta"):
    path = getattr(bundle, field)
    print(f"{field:14} {Path(path).name if path else None}")

## Export Results

Results from all four functions can be exported to JSON format for downstream processing or archival. This is useful for building reproducible pipelines where intermediate results need to be saved between steps.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

fetch_result.export("ncbi_efetch_results", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'ncbi_efetch_results.json'}")

summary_result.export("ncbi_gene_summary", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'ncbi_gene_summary.json'}")